# 02 — Cleaning, features and patient-level split

**Goal:** validate on the real data the pipeline defined in `src/readmission/`: row exclusions,
derived features, the train/test split by patient and the preprocessing.

In [1]:
from readmission import config
from readmission.data import add_binary_target, load_raw
from readmission.features import build_preprocessor, get_feature_columns, prepare
from readmission.split import patient_split

df = prepare(add_binary_target(load_raw()))
numeric, categorical = get_feature_columns(df)
print(df.shape, len(numeric), len(categorical))

(99343, 56) 10 24


In [2]:
train, test = patient_split(df)
print(len(train), len(test))
print(train[config.TARGET].mean(), test[config.TARGET].mean())
print(set(train[config.GROUP_COL]) & set(test[config.GROUP_COL]))  # must be empty

79541 19802
0.11438126249355678 0.11190788809211191
set()


In [3]:
df[numeric + categorical].nunique().sort_values().head(10)  # any constant columns left?

diabetesMed            2
change                 2
gender                 3
glyburide-metformin    4
rosiglitazone          4
pioglitazone           4
acarbose               4
insulin                4
repaglinide            4
metformin              4
dtype: int64

In [4]:
# 1. No nulls, sensible categories
print(df[numeric + categorical].isna().sum().sum())  # expected: 0
for col in ["admission_type_id", "discharge_disposition_id", "diag_1_group"]:
    print(df[col].value_counts(normalize=True).round(3), "\n")

0
admission_type_id
1          0.527
3          0.188
2          0.183
Unknown    0.102
7          0.000
4          0.000
Name: proportion, dtype: float64 

discharge_disposition_id
1          0.606
3          0.140
6          0.130
Unknown    0.047
2          0.021
22         0.020
5          0.012
4          0.008
7          0.006
23         0.004
28         0.001
8          0.001
15         0.001
24         0.000
9          0.000
17         0.000
16         0.000
10         0.000
27         0.000
12         0.000
Name: proportion, dtype: float64 

diag_1_group
Circulatory        0.299
Other              0.179
Respiratory        0.140
Digestive          0.094
Diabetes           0.087
Injury             0.069
Genitourinary      0.050
Musculoskeletal    0.050
Neoplasms          0.032
Unknown            0.000
Name: proportion, dtype: float64 



In [5]:
# 2. Near-constant categorical columns
dominant = df[categorical].apply(lambda s: s.value_counts(normalize=True).iloc[0])
dominant.sort_values(ascending=False).head(12).round(4)

acarbose               0.9969
nateglinide            0.9931
glyburide-metformin    0.9930
repaglinide            0.9847
glimepiride            0.9484
max_glu_serum          0.9481
rosiglitazone          0.9365
pioglitazone           0.9270
glyburide              0.8941
glipizide              0.8739
A1Cresult              0.8305
metformin              0.8002
dtype: float64

In [6]:
# 3. Preprocessor on real data (fitted on train ONLY)
pre = build_preprocessor(numeric, categorical)
X_train = pre.fit_transform(train[numeric + categorical])
X_test = pre.transform(test[numeric + categorical])
print(X_train.shape, X_test.shape, X_train.isna().sum().sum(), X_test.isna().sum().sum())

(79541, 175) (19802, 175) 0 0


## Conclusions
- After excluding death/hospice discharges: 99,343 encounters (11.4% readmitted in <30 days).
- Patient-level split: 79,541 train / 19,802 test, no patient in both sets; readmission rate
  11.4% vs 11.2%.
- 12 near-constant medication columns dropped (≥99.9% of rows share one value).
- 24 categorical and 10 numeric features; ID and sensitive columns kept aside for the split and
  the subgroup analysis.
- The preprocessor is fitted on train only and applied to test (no information leaks from test).